In [7]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

### 1. Data Preprocessing

In [8]:
# 앵커 유저 데이터 로드
split_name = "train"
file_path = f"../dataset/{split_name}.csv"
df = pd.read_csv(file_path)

# 비재무(Source) 범주형 컬럼 리스트 정의
categorical_cols = [
    'OCCAT1', 'OCCAT2', 'INDCAT', 'LF', 'HOUSECL', 'EDCL', 'EDUC', 
    'AGECL', 'LIFECL', 'FAMSTRUCT', 'KIDS', 'MARRIED', 'EXPENSHILO', 'WSAVED', 
    'SAVRES1', 'SAVRES2', 'SAVRES3', 'SAVRES4', 'SAVRES5', 
    'SAVRES6', 'SAVRES7', 'SAVRES8', 'SAVRES9'
]

# FIN(총 금융자산)이 0 초과인 유저만 필터링
df_filtered = df[df['FIN'] > 0].copy()

# 결측치가 있는 범주형 컬럼 확인
missing_counts = df_filtered[categorical_cols].isna().sum()
missing_cols = missing_counts[missing_counts > 0]
if len(missing_cols) > 0:
    print("결측값이 있는 범주형 컬럼:")
    print(missing_cols)
else:
    print("결측값이 있는 범주형 컬럼 없음")

# 제거할 행의 CASEID 추출
rows_before = len(df_filtered)
if 'CASEID' in df_filtered.columns:
    dropped_case_ids = df_filtered[df_filtered[categorical_cols].isna().any(axis=1)]['CASEID'].tolist()
    if dropped_case_ids:
        print(f"삭제된 행의 CASEID ({len(dropped_case_ids)}개): {dropped_case_ids}")
    else:
        print("삭제된 행의 CASEID 없음")
else:
    dropped_case_ids = []
    print("CASEID 컬럼이 없어 삭제된 행 ID를 출력할 수 없습니다.")

# 실제 제거 수행
rows_before = len(df_filtered)
df_final = df_filtered.dropna(subset=categorical_cols).reset_index(drop=True).copy()
rows_after = len(df_final)
print(f"삭제된 행 수: {rows_before - rows_after}개 (범주형 결측값 포함)")

# 1-1. Source Domain (비재무 데이터) 전처리

# 모든 카테고리 변수를 float에서 int형으로 안전하게 변환 (PyTorch Embedding은 정수형을 받음)
for col in categorical_cols:
    df_final[col] = df_final[col].astype(int)

# 각 컬럼별 '최대값 + 1'을 구해서 임베딩 클래스 개수(num_classes)를 동적 계산
# (예: 값이 1,2,3,4 이면 PyTorch는 0~4 인덱스를 커버해야 하므로 5칸의 자리가 필요함)
categorical_cardinalities = [df_final[col].max() + 1 for col in categorical_cols]

# 모델 입력용 PyTorch 텐서로 변환
x_cat_tensor = torch.tensor(df_final[categorical_cols].values, dtype=torch.long)

# 1-2. Target Domain (재무 데이터) 전처리
# 자산군 그룹화 및 비율 계산
df_final['RATIO_CASH'] = (df_final['LIQ'] + df_final['CDS']) / df_final['FIN']
df_final['RATIO_RISK'] = df_final['EQUITY'] / df_final['FIN']
df_final['RATIO_BOND'] = df_final['BOND'] / df_final['FIN']
df_final['RATIO_PENSION'] = df_final['RETQLIQ'] / df_final['FIN']

# 4차원 비율 벡터만 추출 후 NaN 처리
ratio_columns = ['RATIO_CASH', 'RATIO_RISK', 'RATIO_BOND', 'RATIO_PENSION']
df_ratios = df_final[ratio_columns].fillna(0)

# 소수점 오차 방지를 위해 행(User) 단위로 다시 한 번 비율 L1 정규화 (합을 무조건 1로!)
df_ratios = df_ratios.div(df_ratios.sum(axis=1), axis=0).fillna(0)

# 모델 입력용 PyTorch 텐서로 변환
x_ratio_tensor = torch.tensor(df_ratios.values, dtype=torch.float32)

labels = df_final['RISK_LABEL'].astype(int).values

if labels.min() == 1:
    labels = labels - 1
    print("Adjusted labels from 1-5 to 0-4.")

labels_tensor = torch.tensor(labels, dtype=torch.long)

print(f"\n전처리 완료: 총 {len(df_final)}명의 유저 데이터 준비됨.")
print(f" - 비재무 텐서 크기: {x_cat_tensor.shape} (유저수, 23개 범주형 컬럼)")
print(f" - 재무 텐서 크기: {x_ratio_tensor.shape} (유저수, 4개 자산군 비율)")
print(f" - 라벨 텐서 크기: {labels_tensor.shape}\n")


결측값이 있는 범주형 컬럼:
EDUC    1
dtype: int64
삭제된 행의 CASEID (1개): [200306]
삭제된 행 수: 1개 (범주형 결측값 포함)
Adjusted labels from 1-5 to 0-4.

전처리 완료: 총 13207명의 유저 데이터 준비됨.
 - 비재무 텐서 크기: torch.Size([13207, 23]) (유저수, 23개 범주형 컬럼)
 - 재무 텐서 크기: torch.Size([13207, 4]) (유저수, 4개 자산군 비율)
 - 라벨 텐서 크기: torch.Size([13207])



### 2. Model Definition

In [9]:
from models import SourceEncoder, TargetEncoder

output_latent_dim = 128 # 두 벡터가 만나는 중간 공간의 차원

### 3. Forward Pass Test

In [10]:
# 모델 초기화
source_encoder = SourceEncoder(categorical_cardinalities, embed_dim=16, output_dim=output_latent_dim)
target_encoder = TargetEncoder(input_dim=4, output_dim=output_latent_dim)

# 전체 데이터 통과시켜보기 (메모리 절약을 위해 실제 학습 시에는 DataLoader로 Mini-batch 처리 필요)
source_encoder.eval()
target_encoder.eval()

with torch.no_grad():
    _, z_NF = source_encoder(x_cat_tensor)
    _, z_F = target_encoder(x_ratio_tensor)

print("모델 통과 결과:")
print(f" - 추출된 비재무 잠재 벡터 (z_NF): {z_NF.shape}")
print(f" - 추출된 재무 잠재 벡터 (z_F): {z_F.shape}")

모델 통과 결과:
 - 추출된 비재무 잠재 벡터 (z_NF): torch.Size([13207, 64])
 - 추출된 재무 잠재 벡터 (z_F): torch.Size([13207, 64])


### 4. Save Tensor

In [11]:
torch.save(x_cat_tensor, f'../dataset/processed/{split_name}_x_cat_tensor.pt')
torch.save(x_ratio_tensor, f'../dataset/processed/{split_name}_x_ratio_tensor.pt')
torch.save(labels_tensor, f'../dataset/processed/{split_name}_labels_tensor.pt')
torch.save(torch.tensor(categorical_cardinalities), f'../dataset/processed/{split_name}_cardinalities.pt')
print("데이터 저장 완료")

데이터 저장 완료


In [12]:
assert len(x_cat_tensor) == len(x_ratio_tensor) == len(labels_tensor)
print("텐서 길이 정합성 확인 완료:", len(labels_tensor))

텐서 길이 정합성 확인 완료: 13207
